In [1]:
from scipy.io import loadmat
from scipy.signal import butter, filtfilt, find_peaks, firwin, medfilt
import numpy as np
import pandas as pd
import sqlite3
import os
from pathlib import Path
import matplotlib.pyplot as plt

In [2]:
conn = sqlite3.connect(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\Recordings_FH.db")
sql = """
SELECT r.Animal_Id, r.Cell_Id, r.Folderpath, r.Condition, r.exp_type
FROM Recordings as r
WHERE exp_type = 'juxta'
AND r.Condition = 'ramp'
AND USE = 1
ORDER BY r.Cell_Id DESC"""

datarow = pd.read_sql_query(sql, conn)
data_row = datarow.head(1)
conn.close()

In [3]:
def TriggRasterPY(
        tone_triggers,
        spkT,
        SR = 25000,
        halftime = 0.6,
        nbins = 100
):
    """
    This is a replica of the TriggRaster.m function.
    It did not implement the shuffling, since it's not used for me anyways.
    There is a minor discrepancy between the rate values of roughly 1 Hz.
    This comes to play because Matlab treats the value of time2bin differently
    and rounds it weirdly.
    """

    spkT_samples = np.round(spkT*SR)
    halfsamples = round(halftime*SR)
    chunks = np.zeros((len(tone_triggers), (halfsamples*2)+1))

    for idx, ttrig in enumerate(tone_triggers):
        c = np.linspace(ttrig-halfsamples,ttrig+halfsamples,(halfsamples*2)+1, dtype=int) # time in samples. Needs to match matlab
        isCell = np.isin(c, spkT_samples) # indices corresponding to the window of c; this is matlab idx -1
        chunks[idx,:] = isCell


    timechunk = np.linspace(-halftime, halftime, (halfsamples*2)+1)

    edges = np.round(np.linspace(0,len(timechunk), nbins)) # edges are used for binning so i need indices from 0 again
    rows, cols = np.nonzero(chunks)
    sorting_columns_idx = np.argsort(cols) # sort the samples of spiketimes in ascending order
    cols = cols[sorting_columns_idx]
    rows = rows[sorting_columns_idx] # apply the sorting to the rows array as well
    counts, edges = np.histogram(cols, bins=edges)
    # mimic the medfilt1
    edges2plot = np.concatenate([[edges[0]], np.median(np.vstack([edges[:-1], edges[1:]]), axis=0)])
    edges2plot = edges2plot[1::]
    time2scale = np.round(np.median(np.diff(timechunk[np.round(edges2plot).astype(int)])),4)
    ntrial_timebin = len(tone_triggers)*time2scale

    # estimation of firing rate)
    rate = np.divide(counts, ntrial_timebin)
    raster = {
        'raster_times':timechunk[cols]*1000,
        'raster_row':rows,
        'raster_rate':rate,
        'time': np.round(timechunk[np.round(edges2plot).astype(int)]*1000),
        'time_bin': time2scale*1000 # in milliseconds
    }
    return raster

In [ ]:
# def tonestim_output_ramp(data_row):
"""
Port of tonestim_output_ramp.m (ramp condition), returning a
1-row pandas DataFrame. Uses the schema helpers so all columns
exist even if certain data (like pupil) are missing.
"""

# constants
tone_letter = ['a', 'w', 'e', 't']
pupil_sr = 50
ephys_sr = 25000
halftime_spikes = 8        # sec window for TriggRaster
rasterBins = halftime_spikes * 800
halftime_mot = 8           # sec around trigger for pupil/whisk
smooth_pupil_value = 15    # for smoothing pupil signal
mot_samples = halftime_mot * 2 * pupil_sr

print(f"{'':-^100}")
print(f" Animal_Id {data_row['Animal_Id'].iloc[0]}; Cell_Id {int(data_row['Cell_Id'].iloc[0])} being processed ")
print(f"{'':-^100}")

# init shared vars
tone_onset = None
tone_code = None
pupil_psth = None
whisk_psth = None
RasterTimes = dict()
RasterRows = dict()
RasterRate = dict()
whisk_avg = dict()
pupil_avg = dict()
all_whisk = None
mot_avg = None
trigger_time = None

########################################################
# load trigger/ephys info for juxta OR behav experiment
########################################################
if data_row['exp_type'].values == 'juxta':
# load exp_data.mat
    try:
        d_path = Path(data_row['Folderpath'].values[0], 'exp_data.mat')
        print(d_path)
        exp = loadmat(
            d_path,
            struct_as_record=False,
            squeeze_me=False,
            simplify_cells= True  
        )
    except Exception:
        print(f"Missing exp_data.mat for {data_row['Animal_Id']} Cell_Id {data_row['Cell_Id']}")
        # we'll still build an empty row at the end
        spkT = None
    else:
        processed_data = exp.get('processed_data', None)
        raw_data = exp.get('raw_data', None)
        exp_info = exp.get('info', None)
        analysis = exp.get('analysis', None)

    if 'spike_sorting_data' not in processed_data:
        KeyError(format('Cell_id {} processed_data empty', data_row.Cell_Id.values[0]))

    ## processing
    # spike times
    spkT = processed_data['spike_sorting_data']['spike_times']
    spkT_samples = np.round(spkT * ephys_sr)
    
    # get tone times
    tone_times = raw_data['ephys_data']['keyboard_times'][:]
    # get tone codes
    tone_codes = raw_data['ephys_data']['keyboard_codes'][:,0]

    # sampling rate
    ephys_sr = round(raw_data['ephys_data']['sampling_rate'])

    # get time calls in ephys samples
    tone_onset = np.zeros(len(tone_times))
    for idx, _ in enumerate(tone_times):
        tmp = np.abs(raw_data['ephys_data']['ephy_times'] - tone_times[idx])
        min_idx = np.argmin(tmp)
        tone_onset[idx] = min_idx+1

    # obtain psth information
    for idx, char in enumerate(tone_letter):
        this_code = ord(char) # translate character to ascii
        this_codes = np.flatnonzero(this_code == tone_codes) #find occurrences of ascii character in all stimulations

        tone_triggers = tone_onset[this_codes] # filter for code trigger timings in samples

        raster = TriggRasterPY(
            tone_triggers,
            spkT,
            ephys_sr,
            halftime_spikes,
            rasterBins,
        )
        RasterRows[char] = raster['raster_row']
        RasterTimes[char] = raster['raster_times']
        RasterRate[char] = raster['raster_rate']

# Pupil Data
try:
    pout = loadmat(
        r"\\172.25.250.112\burgalossi\lab share\Data\Florian\ADN\FH8Soso\analysis\Data20\ramp\pupil_data.mat",
        struct_as_record=False,
        squeeze_me=False,
        simplify_cells= True  
    )
    pout = pout.get('pupil_out')
except:
    warn_string = "Animal {animal_id}, Cell_Id {cell_id}, Condition {condition} no pupil data".format(
        animal_id = data_row['Animal_Id'],
        cell_id= data_row['Cell_Id'],
        condition=data_row['Condition']
        )
    Warning(warn_string)
    #return

if 'pupil_times' not in pout:
    stri = "No pupil times in Animal {}, Cell_Id {}, Condition {}".format(data_row['Animal_Id'], data_row['Cell_Id'], data_row['Condition'])
    Warning(stri)

whisk_motion = pout['motion']

# remove artifacts in motion
max_min = np.max(whisk_motion) - np.min(whisk_motion)
mm_std_ratio = max_min/np.std(whisk_motion)

if mm_std_ratio > 7:
    threshold = np.percentile(whisk_motion, 99.7)
    whisk_motion[whisk_motion > threshold] = np.nan

def smooth(x, span):
    span = int(span)
    kernel = np.ones(span) /span
    return np.convolve(x, kernel, mode='same')

def but_filter(x, cutoff, fs, order=3):
    b,a = butter(order, cutoff , btype='bandpass')
    return filtfilt(b,a,x)

# smooth and filter pupil
pupil_area = smooth(pout['pupil_area'], smooth_pupil_value)
pupil_area = but_filter(pupil_area,[.1/(pupil_sr/2), 1/(pupil_sr/2)],pupil_sr)

def normalize(x):
    x = np.asarray(x)
    return (x - x.min()) /(x.max() - x.min())

pupil_area = normalize(pupil_area)
whisk_motion = normalize(whisk_motion)

pupil_psth = dict()
whisk_psth = dict()
for idx, tone in enumerate(tone_letter):
    this_code = ord(tone)
    this_codes = np.flatnonzero(this_code==tone_codes)

    tone_triggers = tone_times[this_codes]

    # triggers to time
    if tone_triggers.max() > 100000:
        tone_triggers = np.divide(tone_triggers, ephys_sr)
    
    pupil_chunks = np.zeros((len(tone_triggers), mot_samples))    
    whisk_chunks = np.zeros((len(tone_triggers), mot_samples))    

    #loop across stimuli
    for jdx, tt in enumerate(tone_triggers):
        chunk_win = np.array([-halftime_mot, halftime_mot]) + tt

        if chunk_win.min() < 0 or chunk_win.max() > pout['pupil_times'][-1]:
            continue

        my_window = np.flatnonzero((pout['pupil_times'] > chunk_win[0]) & (pout['pupil_times'] <= chunk_win[1]))
        if len(my_window) == mot_samples -1:
            np.append(my_window, my_window[-1]+1)

        if len(my_window) > mot_samples: print(tt); continue;
        pupil_chunks[jdx, :] = pupil_area[my_window]
        whisk_chunks[jdx, :] = whisk_motion[my_window]

    # motion trigger time
    trigger_time = np.linspace(-halftime_mot, halftime_mot, mot_samples)

    pupil_psth[tone] = pupil_chunks
    whisk_psth[tone] = whisk_chunks
    pupil_avg[tone] = np.nanmean(pupil_chunks, axis=0)
    whisk_avg[tone] = np.nanmean(whisk_chunks, axis=0)
        
            
comb_table = dict()
comb_table['trigger_time'] = trigger_time[0]
comb_table['pupil_psth'] = pupil_psth
comb_table['whisk_psth'] = whisk_psth
comb_table['pupil_avg'] = pupil_avg
comb_table['whisk_avg'] = whisk_avg



----------------------------------------------------------------------------------------------------
 Animal_Id FH8soso; Cell_Id 20 being processed 
----------------------------------------------------------------------------------------------------
\\172.25.250.112\burgalossi\lab share\Data\Florian\ADN\FH8Soso\analysis\Data20\ramp\exp_data.mat


In [26]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource
from bokeh.layouts import column

def show_bokeh_inline(data, kind="line", title="Interactive Plot"):
    """
    Show an interactive Bokeh plot inline (no server).
    Great for notebooks and avoids event loop issues entirely.
    """
    try:
        output_notebook()
    except Exception:
        pass

    if isinstance(data, dict):
        x = data.get("x")
        y = data.get("y")
        if x is None or y is None:
            raise ValueError("When passing a dict, include keys 'x' and 'y'.")
    else:
        y = list(data)
        x = list(range(len(y)))

    source = ColumnDataSource(dict(x=x, y=y))
    p = figure(title=title, width=900, height=450, tools="pan,wheel_zoom,box_zoom,reset,hover,save")

    if kind == "line":
        p.line("x", "y", source=source)
    elif kind == "scatter":
        p.circle("x", "y", source=source, size=6)
    elif kind == "step":
        p.step("x", "y", source=source, mode="center")
    else:
        raise ValueError(f"Unsupported kind='{kind}'. Use 'line', 'scatter', or 'step'.")

    show(column(p))

